In [7]:
"""
Enhanced Gene-to-m/z Spatial Pattern Matching Pipeline
=======================================================

This enhanced version implements:
1. Attention weight extraction from GAT layers
2. Spatial importance/saliency maps from attention
3. Region-aware embedding reweighting
4. Global spatial signatures for genes and m/z features
5. Deep embedding similarity instead of raw correlations
6. Saliency map visualization and interpretation
7. Soft masking during training based on region importance
8. Sample-specific spatial signatures for cross-sample matching

Author: Enhanced from original pipeline
"""

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, global_mean_pool, global_add_pool
from torch_geometric.utils import softmax
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial import Delaunay, distance_matrix
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional, Union
import pandas as pd
import os
import warnings
from collections import defaultdict
from dataclasses import dataclass
import json

warnings.filterwarnings('ignore')


# =============================================================================
# DATA CONFIGURATION
# =============================================================================

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

# Target genes for AAD group
AAD_TARGET_GENES = [
    'Mapt', 'Thy1', 'Pmch', 'Atp1a3', 'Rac1', 'Rsrp1', 'Snhg11', 'Tubb4a',
    'Rasgrf1', 'Hsp90ab1', 'Elavl3', 'App', 'Syp', 'AC149090.1',
    'Aplp1', 'Apoe', 'Meg3', 'Gnas', 'Pkm'
]


In [8]:


# =============================================================================
# DATA CLASSES FOR STRUCTURED OUTPUTS
# =============================================================================

@dataclass
class AttentionOutput:
    """Container for attention-related outputs"""
    node_embeddings: torch.Tensor  # Shape: [num_nodes, embedding_dim]
    attention_weights: List[torch.Tensor]  # Per-layer attention weights
    node_importance: torch.Tensor  # Shape: [num_nodes] - aggregated importance
    weighted_embeddings: torch.Tensor  # Shape: [num_nodes, embedding_dim]
    global_embedding: torch.Tensor  # Shape: [embedding_dim]


@dataclass
class SpatialSignature:
    """Spatial signature for a gene or m/z feature"""
    sample_id: str
    feature_name: str
    feature_type: str  # 'gene' or 'mz'
    global_embedding: np.ndarray
    node_embeddings: np.ndarray
    node_importance: np.ndarray
    coordinates: np.ndarray
    raw_values: np.ndarray


@dataclass
class MatchResult:
    """Result of matching a gene to m/z features"""
    gene: str
    rna_sample: str
    mz_feature: str
    msi_sample: str
    embedding_cosine: float
    embedding_euclidean: float
    attention_overlap: float  # How much important regions overlap
    raw_pearson: float
    raw_spearman: float
    combined_score: float


In [9]:
# =============================================================================
# ENHANCED GAT MODEL WITH ATTENTION EXTRACTION
# =============================================================================

class AttentionGATConv(GATConv):
    """
    Extended GATConv that stores attention weights for later extraction.
    This allows us to build spatial importance maps.
    """
    
    def __init__(self, *args, **kwargs):
        # Ensure we return attention weights
        kwargs['add_self_loops'] = kwargs.get('add_self_loops', True)
        super().__init__(*args, **kwargs)
        self.stored_attention = None
        self.stored_edge_index = None
    
    def forward(self, x, edge_index, return_attention_weights=True):
        """Forward pass that stores attention weights"""
        # Call parent forward with attention weight return
        out = super().forward(x, edge_index, return_attention_weights=return_attention_weights)
        
        if return_attention_weights:
            out, (edge_index_out, attention_weights) = out
            self.stored_attention = attention_weights
            self.stored_edge_index = edge_index_out
            return out
        return out


class SpatialAttentionGNN(nn.Module):
    """
    Enhanced GNN with attention weight extraction and spatial importance mapping.
    
    Key Features:
    1. Multi-head attention with extractable weights
    2. Layer-wise attention aggregation
    3. Node importance scoring from attention patterns
    4. Region-aware embedding generation
    5. Global spatial signature pooling
    
    This version uses a flexible input projection layer that can handle
    any input dimension, making it suitable for both multi-feature training
    and single-feature inference.
    """
    
    def __init__(
        self,
        input_dim: int = None,  # Can be None for flexible input
        hidden_dim: int = 128,
        embedding_dim: int = 64,
        num_layers: int = 3,
        num_heads: int = 4,
        dropout: float = 0.3,
        use_importance_weighting: bool = True,
        projection_dim: int = 64  # Project any input to this dimension first
    ):
        super(SpatialAttentionGNN, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.use_importance_weighting = use_importance_weighting
        self.projection_dim = projection_dim
        
        # Store projections for different input dims (for flexibility)
        self.input_projections = nn.ModuleDict()
        
        # If input_dim is provided, create default projection
        if input_dim is not None:
            self.input_projections[str(input_dim)] = nn.Linear(input_dim, projection_dim)
        
        # GAT layers with attention extraction - now use projection_dim as input
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        # First layer: projection_dim -> hidden_dim * num_heads
        self.convs.append(AttentionGATConv(
            projection_dim, hidden_dim, heads=num_heads, concat=True, dropout=dropout
        ))
        self.batch_norms.append(nn.BatchNorm1d(hidden_dim * num_heads))
        
        # Hidden layers: hidden_dim * num_heads -> hidden_dim * num_heads
        for _ in range(num_layers - 2):
            self.convs.append(AttentionGATConv(
                hidden_dim * num_heads, hidden_dim, heads=num_heads, 
                concat=True, dropout=dropout
            ))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim * num_heads))
        
        # Output layer: hidden_dim * num_heads -> embedding_dim
        self.convs.append(AttentionGATConv(
            hidden_dim * num_heads, embedding_dim, heads=1, 
            concat=False, dropout=dropout
        ))
        
        # Importance scoring network
        self.importance_net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        
        # Attention pooling for global embedding
        self.attention_pool = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def _get_or_create_projection(self, input_dim: int) -> nn.Linear:
        """Get or create input projection for given input dimension"""
        dim_key = str(input_dim)
        
        if dim_key not in self.input_projections:
            # Create new projection layer for this input dimension
            device = next(self.parameters()).device
            proj = nn.Linear(input_dim, self.projection_dim).to(device)
            # Initialize with Xavier initialization
            nn.init.xavier_uniform_(proj.weight)
            nn.init.zeros_(proj.bias)
            self.input_projections[dim_key] = proj
        
        return self.input_projections[dim_key]
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, 
                batch: Optional[torch.Tensor] = None) -> AttentionOutput:
        """
        Forward pass with full attention extraction.
        
        Args:
            x: Node features [num_nodes, input_dim] - can be any input dimension
            edge_index: Graph connectivity [2, num_edges]
            batch: Batch assignment for multiple graphs [num_nodes]
            
        Returns:
            AttentionOutput containing embeddings, attention, and importance
        """
        attention_weights_per_layer = []
        
        # Get the appropriate projection layer for this input dimension
        input_dim = x.size(1)
        projection = self._get_or_create_projection(input_dim)
        
        # Project input to fixed dimension
        h = projection(x)
        
        # Forward through GAT layers
        for i in range(len(self.convs) - 1):
            h = self.convs[i](h, edge_index, return_attention_weights=True)
            attention_weights_per_layer.append(self.convs[i].stored_attention)
            h = self.batch_norms[i](h)
            h = F.elu(h)
            h = self.dropout(h)
        
        # Final layer
        node_embeddings = self.convs[-1](h, edge_index, return_attention_weights=True)
        attention_weights_per_layer.append(self.convs[-1].stored_attention)
        
        # Compute node importance from attention patterns
        node_importance = self._compute_node_importance(
            attention_weights_per_layer, edge_index, x.size(0)
        )
        
        # Apply importance weighting to embeddings
        if self.use_importance_weighting:
            # Normalize importance to prevent scale issues
            importance_normalized = node_importance / (node_importance.sum() + 1e-8) * x.size(0)
            weighted_embeddings = node_embeddings * importance_normalized.unsqueeze(-1)
        else:
            weighted_embeddings = node_embeddings
        
        # Generate global embedding via attention pooling
        global_embedding = self._attention_pool(weighted_embeddings, batch)
        
        return AttentionOutput(
            node_embeddings=node_embeddings,
            attention_weights=attention_weights_per_layer,
            node_importance=node_importance,
            weighted_embeddings=weighted_embeddings,
            global_embedding=global_embedding
        )
    
    def _compute_node_importance(
        self, 
        attention_weights: List[torch.Tensor],
        edge_index: torch.Tensor,
        num_nodes: int
    ) -> torch.Tensor:
        """
        Aggregate attention weights across layers to compute node importance.
        
        This creates a saliency score for each node based on how much attention
        it receives from its neighbors across all GAT layers.
        """
        device = edge_index.device
        importance = torch.zeros(num_nodes, device=device)
        
        for layer_idx, attn in enumerate(attention_weights):
            if attn is None:
                continue
            
            # Get the edge index for this layer
            layer_edge_index = self.convs[layer_idx].stored_edge_index
            if layer_edge_index is None:
                continue
            
            # Attention weights shape: [num_edges, num_heads] or [num_edges]
            if attn.dim() > 1:
                attn_mean = attn.mean(dim=-1)  # Average over heads
            else:
                attn_mean = attn
            
            # Aggregate incoming attention for each node
            target_nodes = layer_edge_index[1]  # Target nodes receive attention
            
            # Use scatter_add to aggregate
            importance.scatter_add_(0, target_nodes, attn_mean)
        
        # Normalize importance scores
        importance = importance / (len(attention_weights) + 1e-8)
        importance = (importance - importance.min()) / (importance.max() - importance.min() + 1e-8)
        
        return importance
    
    def _attention_pool(
        self, 
        embeddings: torch.Tensor, 
        batch: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Attention-weighted pooling to generate global embedding.
        
        Similar to how FaceNet generates a single embedding for an entire face,
        this pools all node embeddings into one spatial signature vector.
        """
        # Compute attention scores
        attn_scores = self.attention_pool(embeddings)  # [num_nodes, 1]
        
        if batch is None:
            # Single graph - softmax over all nodes
            attn_weights = F.softmax(attn_scores, dim=0)
            global_emb = (attn_weights * embeddings).sum(dim=0)
        else:
            # Multiple graphs - softmax within each graph
            attn_weights = softmax(attn_scores.squeeze(-1), batch)
            global_emb = global_add_pool(attn_weights.unsqueeze(-1) * embeddings, batch)
        
        return global_emb


class RegionAwareLoss(nn.Module):
    """
    Loss function that incorporates region importance for training.
    
    This encourages the model to:
    1. Learn spatially consistent patterns
    2. Focus on biologically relevant regions
    3. Ignore noisy background areas
    """
    
    def __init__(
        self,
        spatial_weight: float = 1.0,
        consistency_weight: float = 1.0,
        importance_weight: float = 0.5
    ):
        super().__init__()
        self.spatial_weight = spatial_weight
        self.consistency_weight = consistency_weight
        self.importance_weight = importance_weight
    
    def forward(
        self,
        output: AttentionOutput,
        pos: torch.Tensor,
        edge_index: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute region-aware training loss.
        
        Args:
            output: AttentionOutput from the model
            pos: Spatial coordinates [num_nodes, 2]
            edge_index: Graph connectivity
            
        Returns:
            Combined loss value
        """
        embeddings = output.weighted_embeddings
        importance = output.node_importance
        
        # 1. Spatial consistency loss - nearby nodes should have similar embeddings
        spatial_loss = self._spatial_consistency_loss(embeddings, pos, importance)
        
        # 2. Embedding distance should correlate with spatial distance
        distance_loss = self._distance_preservation_loss(embeddings, pos)
        
        # 3. Importance regularization - encourage meaningful importance distribution
        importance_loss = self._importance_regularization(importance)
        
        total_loss = (
            self.spatial_weight * spatial_loss +
            self.consistency_weight * distance_loss +
            self.importance_weight * importance_loss
        )
        
        return total_loss
    
    def _spatial_consistency_loss(
        self, 
        embeddings: torch.Tensor, 
        pos: torch.Tensor,
        importance: torch.Tensor
    ) -> torch.Tensor:
        """Nearby nodes should have similar embeddings, weighted by importance"""
        # Sample pairs to avoid O(n^2) computation
        num_nodes = embeddings.size(0)
        num_samples = min(5000, num_nodes * 10)
        
        idx1 = torch.randint(0, num_nodes, (num_samples,), device=embeddings.device)
        idx2 = torch.randint(0, num_nodes, (num_samples,), device=embeddings.device)
        
        # Spatial distances
        spatial_dist = torch.norm(pos[idx1] - pos[idx2], dim=1)
        
        # Embedding distances
        emb_dist = torch.norm(embeddings[idx1] - embeddings[idx2], dim=1)
        
        # Weight by importance of both nodes
        pair_importance = (importance[idx1] + importance[idx2]) / 2
        
        # Loss: embedding distance should be small for spatially close nodes
        # Use Gaussian kernel for spatial proximity
        sigma = spatial_dist.median()
        spatial_weight = torch.exp(-spatial_dist ** 2 / (2 * sigma ** 2 + 1e-8))
        
        loss = (spatial_weight * pair_importance * emb_dist).mean()
        
        return loss
    
    def _distance_preservation_loss(
        self, 
        embeddings: torch.Tensor, 
        pos: torch.Tensor
    ) -> torch.Tensor:
        """Embedding distances should correlate with spatial distances"""
        num_nodes = embeddings.size(0)
        num_samples = min(2000, num_nodes * 5)
        
        idx1 = torch.randint(0, num_nodes, (num_samples,), device=embeddings.device)
        idx2 = torch.randint(0, num_nodes, (num_samples,), device=embeddings.device)
        
        spatial_dist = torch.norm(pos[idx1] - pos[idx2], dim=1)
        emb_dist = torch.norm(embeddings[idx1] - embeddings[idx2], dim=1)
        
        # Normalize to [0, 1]
        spatial_dist = spatial_dist / (spatial_dist.max() + 1e-8)
        emb_dist = emb_dist / (emb_dist.max() + 1e-8)
        
        loss = F.mse_loss(emb_dist, spatial_dist)
        
        return loss
    
    def _importance_regularization(self, importance: torch.Tensor) -> torch.Tensor:
        """Encourage non-trivial importance distribution"""
        # Entropy regularization - avoid uniform or degenerate distributions
        p = importance / (importance.sum() + 1e-8)
        entropy = -(p * torch.log(p + 1e-8)).sum()
        
        # Target a moderate entropy (not too uniform, not too peaked)
        target_entropy = np.log(len(importance)) * 0.5
        entropy_loss = (entropy - target_entropy) ** 2
        
        # Also encourage some nodes to be important (avoid all-low importance)
        sparsity_loss = -importance.max()
        
        return entropy_loss + 0.1 * sparsity_loss


In [10]:
# =============================================================================
# ENHANCED GENE-TO-MZ MATCHER
# =============================================================================

class EnhancedGeneToMzMatcher:
    """
    Enhanced matcher using deep spatial embeddings instead of raw correlations.
    
    Key improvements:
    1. Attention-based spatial importance mapping
    2. Region-aware embeddings
    3. Deep embedding similarity
    4. Cross-sample matching via spatial signatures
    5. Saliency visualization
    """
    
    def __init__(
        self,
        output_dir: str = './gene_to_mz_results_enhanced',
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
        embedding_dim: int = 64,
        hidden_dim: int = 128,
        num_layers: int = 3,
        num_heads: int = 4
    ):
        self.device = device
        self.output_dir = output_dir
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_heads = num_heads
        
        # Create output directories
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(os.path.join(output_dir, 'gene_visualizations'), exist_ok=True)
        os.makedirs(os.path.join(output_dir, 'saliency_maps'), exist_ok=True)
        os.makedirs(os.path.join(output_dir, 'embeddings'), exist_ok=True)
        os.makedirs(os.path.join(output_dir, 'attention_analysis'), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.groups = ['YC', 'YAD', 'AC', 'AAD']
        
        # Models for RNA and MSI
        self.rna_model = None
        self.msi_model = None
        
        # Cached spatial signatures
        self.gene_signatures: Dict[str, Dict[str, SpatialSignature]] = defaultdict(dict)
        self.mz_signatures: Dict[str, Dict[str, SpatialSignature]] = defaultdict(dict)
        
    def load_all_data(self):
        """Load all RNA and MSI data"""
        print("Loading RNA-seq data...")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
            else:
                print(f"  Warning: {path} not found")
        
        print("\nLoading MSI data...")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
            else:
                print(f"  Warning: {path} not found")
    
    def check_gene_availability(self) -> Dict[str, Dict[str, bool]]:
        """Check which target genes are available in the data"""
        print("\nChecking gene availability across samples...")
        gene_availability = {}
        
        for gene in AAD_TARGET_GENES:
            gene_availability[gene] = {}
            for sample_id in RNA_SAMPLE_IDS:
                if sample_id in self.rna_data:
                    gene_availability[gene][sample_id] = gene in self.rna_data[sample_id].var_names
                else:
                    gene_availability[gene][sample_id] = False
        
        # Summary
        for gene in AAD_TARGET_GENES:
            available_count = sum(gene_availability[gene].values())
            print(f"  {gene}: {available_count}/{len(RNA_SAMPLE_IDS)} samples")
        
        return gene_availability
    
    def build_spatial_graph(
        self, 
        coords: np.ndarray,
        k_neighbors: int = 6
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Build spatial graph from coordinates using Delaunay triangulation.
        Falls back to k-NN if Delaunay fails.
        """
        try:
            tri = Delaunay(coords)
            edges = set()
            for simplex in tri.simplices:
                for i in range(len(simplex)):
                    for j in range(i + 1, len(simplex)):
                        edges.add(tuple(sorted([simplex[i], simplex[j]])))
            edge_list = list(edges)
        except Exception as e:
            # Fallback to k-NN
            print(f"    Delaunay failed ({e}), using k-NN")
            dist_mat = distance_matrix(coords, coords)
            edge_list = []
            for i in range(len(coords)):
                nearest = np.argsort(dist_mat[i])[1:k_neighbors+1]
                for j in nearest:
                    edge_list.append([i, j])
        
        # Make bidirectional
        edge_index = []
        for edge in edge_list:
            edge_index.append([edge[0], edge[1]])
            edge_index.append([edge[1], edge[0]])
        
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        coords_tensor = torch.tensor(coords, dtype=torch.float32)
        
        return edge_index, coords_tensor
    
    def prepare_graph_data(
        self, 
        coords: np.ndarray, 
        features: np.ndarray,
        smooth_sigma: float = 50.0
    ) -> Data:
        """
        Prepare PyTorch Geometric Data object with optional spatial smoothing.
        """
        # Apply spatial smoothing
        if smooth_sigma > 0:
            features_smooth = self._apply_spatial_smoothing(coords, features, smooth_sigma)
        else:
            features_smooth = features
        
        # Normalize features
        if features_smooth.ndim == 1:
            features_smooth = features_smooth.reshape(-1, 1)
        
        scaler = StandardScaler()
        features_normalized = scaler.fit_transform(features_smooth)
        
        # Build graph
        edge_index, coords_tensor = self.build_spatial_graph(coords)
        
        data = Data(
            x=torch.tensor(features_normalized, dtype=torch.float32),
            edge_index=edge_index,
            pos=coords_tensor
        )
        
        return data
    
    def _apply_spatial_smoothing(
        self, 
        coords: np.ndarray, 
        values: np.ndarray, 
        sigma: float
    ) -> np.ndarray:
        """Apply Gaussian spatial smoothing"""
        if values.ndim == 1:
            values = values.reshape(-1, 1)
        
        smoothed = np.zeros_like(values)
        
        for i in range(len(coords)):
            dists = np.sqrt(np.sum((coords - coords[i])**2, axis=1))
            weights = np.exp(-dists**2 / (2 * sigma**2))
            weights /= weights.sum()
            smoothed[i] = (values.T @ weights).T if values.ndim > 1 else values.flatten() @ weights
        
        return smoothed
    
    def train_spatial_model(
        self,
        data_dict: Dict[str, Data],
        model_type: str = 'rna',
        epochs: int = 150,
        lr: float = 0.001
    ) -> SpatialAttentionGNN:
        """
        Train the spatial attention GNN model.
        
        Args:
            data_dict: Dictionary mapping sample_id to Data objects
            model_type: 'rna' or 'msi'
            epochs: Number of training epochs
            lr: Learning rate
            
        Returns:
            Trained model
        """
        print(f"\nTraining {model_type.upper()} spatial model...")
        
        # Get input dimension from first sample (for initial projection)
        first_data = list(data_dict.values())[0]
        input_dim = first_data.x.size(1)
        
        # Initialize model with flexible input (input_dim used for initial projection)
        model = SpatialAttentionGNN(
            input_dim=input_dim,  # Will create initial projection for training data
            hidden_dim=self.hidden_dim,
            embedding_dim=self.embedding_dim,
            num_layers=self.num_layers,
            num_heads=self.num_heads,
            projection_dim=64  # Fixed projection dimension
        ).to(self.device)
        
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
        loss_fn = RegionAwareLoss()
        
        model.train()
        loss_history = []
        
        for epoch in range(epochs):
            total_loss = 0
            
            for sample_id, data in data_dict.items():
                data = data.to(self.device)
                optimizer.zero_grad()
                
                output = model(data.x, data.edge_index)
                loss = loss_fn(output, data.pos, data.edge_index)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                total_loss += loss.item()
            
            scheduler.step()
            avg_loss = total_loss / len(data_dict)
            loss_history.append(avg_loss)
            
            if (epoch + 1) % 25 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
        
        # Plot loss curve
        self._plot_loss_curve(loss_history, model_type)
        
        return model
    
    def _plot_loss_curve(self, loss_history: List[float], model_type: str):
        """Plot and save training loss curve"""
        plt.figure(figsize=(10, 6))
        plt.plot(loss_history)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_type.upper()} Model Training Loss')
        plt.grid(True, alpha=0.3)
        plt.savefig(
            os.path.join(self.output_dir, f'{model_type}_training_loss.png'),
            dpi=150, bbox_inches='tight'
        )
        plt.close()
    
    def extract_gene_spatial_signature(
        self,
        sample_id: str,
        gene: str,
        model: SpatialAttentionGNN,
        smooth_sigma: float = 50.0
    ) -> Optional[SpatialSignature]:
        """
        Extract spatial signature for a specific gene using the trained model.
        
        This is the equivalent of generating a "face embedding" for the gene's
        spatial pattern.
        """
        if sample_id not in self.rna_data:
            return None
        
        adata = self.rna_data[sample_id]
        
        if gene not in adata.var_names:
            return None
        
        # Get coordinates
        coords = np.column_stack([
            adata.obs['x_um'].values,
            adata.obs['y_um'].values
        ])
        
        # Get gene expression
        gene_idx = list(adata.var_names).index(gene)
        if hasattr(adata.X, 'toarray'):
            gene_expr = adata.X[:, gene_idx].toarray().flatten()
        else:
            gene_expr = adata.X[:, gene_idx].flatten()
        
        # Prepare graph data
        data = self.prepare_graph_data(coords, gene_expr, smooth_sigma)
        data = data.to(self.device)
        
        # Extract embeddings
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index)
        
        signature = SpatialSignature(
            sample_id=sample_id,
            feature_name=gene,
            feature_type='gene',
            global_embedding=output.global_embedding.cpu().numpy(),
            node_embeddings=output.weighted_embeddings.cpu().numpy(),
            node_importance=output.node_importance.cpu().numpy(),
            coordinates=coords,
            raw_values=gene_expr
        )
        
        # Cache signature
        self.gene_signatures[gene][sample_id] = signature
        
        return signature
    
    def extract_mz_spatial_signatures(
        self,
        sample_id: str,
        model: SpatialAttentionGNN,
        smooth_sigma: float = 50.0,
        batch_size: int = 50
    ) -> Dict[str, SpatialSignature]:
        """
        Extract spatial signatures for all m/z features in a sample.
        
        Args:
            sample_id: MSI sample identifier
            model: Trained spatial attention GNN
            smooth_sigma: Smoothing parameter
            batch_size: Number of m/z features to process at once
            
        Returns:
            Dictionary mapping m/z feature names to their signatures
        """
        if sample_id not in self.msi_data:
            return {}
        
        adata = self.msi_data[sample_id]
        
        # Get coordinates
        coords = np.column_stack([
            adata.obs['x_um'].values,
            adata.obs['y_um'].values
        ])
        
        # Get all m/z intensities
        if hasattr(adata.X, 'toarray'):
            mz_intensities = adata.X.toarray()
        else:
            mz_intensities = adata.X.copy()
        
        mz_features = list(adata.var_names)
        signatures = {}
        
        print(f"  Extracting signatures for {len(mz_features)} m/z features...")
        
        model.eval()
        
        for i in range(0, len(mz_features), batch_size):
            batch_mz = mz_features[i:i+batch_size]
            
            for j, mz_name in enumerate(batch_mz):
                mz_idx = i + j
                mz_values = mz_intensities[:, mz_idx]
                
                # Prepare graph data
                data = self.prepare_graph_data(coords, mz_values, smooth_sigma)
                data = data.to(self.device)
                
                # Extract embeddings
                with torch.no_grad():
                    output = model(data.x, data.edge_index)
                
                signature = SpatialSignature(
                    sample_id=sample_id,
                    feature_name=mz_name,
                    feature_type='mz',
                    global_embedding=output.global_embedding.cpu().numpy(),
                    node_embeddings=output.weighted_embeddings.cpu().numpy(),
                    node_importance=output.node_importance.cpu().numpy(),
                    coordinates=coords,
                    raw_values=mz_values
                )
                
                signatures[mz_name] = signature
                self.mz_signatures[mz_name][sample_id] = signature
            
            if (i + batch_size) % 200 == 0:
                print(f"    Processed {min(i+batch_size, len(mz_features))}/{len(mz_features)}")
        
        return signatures
    
    def compute_embedding_similarity(
        self,
        sig1: SpatialSignature,
        sig2: SpatialSignature
    ) -> Dict[str, float]:
        """
        Compute similarity between two spatial signatures using multiple metrics.
        
        This is the deep learning equivalent of comparing two "face embeddings".
        """
        # Global embedding similarity
        emb1 = sig1.global_embedding.flatten()
        emb2 = sig2.global_embedding.flatten()
        
        # Cosine similarity
        cosine_sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2) + 1e-8)
        
        # Euclidean distance (converted to similarity)
        euclidean_dist = np.linalg.norm(emb1 - emb2)
        euclidean_sim = 1.0 / (1.0 + euclidean_dist)
        
        # Attention overlap - how much the important regions overlap
        attention_overlap = self._compute_attention_overlap(sig1, sig2)
        
        # Raw value correlations for comparison
        raw_pearson, raw_spearman = self._compute_raw_correlations(sig1, sig2)
        
        return {
            'embedding_cosine': cosine_sim,
            'embedding_euclidean': euclidean_sim,
            'attention_overlap': attention_overlap,
            'raw_pearson': raw_pearson,
            'raw_spearman': raw_spearman
        }
    
    def _compute_attention_overlap(
        self,
        sig1: SpatialSignature,
        sig2: SpatialSignature,
        grid_resolution: int = 50
    ) -> float:
        """
        Compute how much the important regions overlap between two signatures.
        
        This tells us if the model is focusing on the same biological regions
        for both features.
        """
        # Interpolate importance maps to common grid
        x_min = min(sig1.coordinates[:, 0].min(), sig2.coordinates[:, 0].min())
        x_max = max(sig1.coordinates[:, 0].max(), sig2.coordinates[:, 0].max())
        y_min = min(sig1.coordinates[:, 1].min(), sig2.coordinates[:, 1].min())
        y_max = max(sig1.coordinates[:, 1].max(), sig2.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_resolution),
            np.linspace(y_min, y_max, grid_resolution)
        )
        
        # Interpolate importance maps
        imp1_grid = griddata(
            sig1.coordinates, sig1.node_importance,
            (grid_x, grid_y), method='linear', fill_value=0
        )
        imp2_grid = griddata(
            sig2.coordinates, sig2.node_importance,
            (grid_x, grid_y), method='linear', fill_value=0
        )
        
        # Normalize
        imp1_grid = imp1_grid / (imp1_grid.max() + 1e-8)
        imp2_grid = imp2_grid / (imp2_grid.max() + 1e-8)
        
        # Compute overlap (similar to IoU for soft masks)
        intersection = np.minimum(imp1_grid, imp2_grid).sum()
        union = np.maximum(imp1_grid, imp2_grid).sum()
        
        overlap = intersection / (union + 1e-8)
        
        return overlap
    
    def _compute_raw_correlations(
        self,
        sig1: SpatialSignature,
        sig2: SpatialSignature,
        grid_resolution: int = 50
    ) -> Tuple[float, float]:
        """
        Compute raw value correlations for comparison with embedding similarity.
        """
        # Interpolate to common grid
        x_min = min(sig1.coordinates[:, 0].min(), sig2.coordinates[:, 0].min())
        x_max = max(sig1.coordinates[:, 0].max(), sig2.coordinates[:, 0].max())
        y_min = min(sig1.coordinates[:, 1].min(), sig2.coordinates[:, 1].min())
        y_max = max(sig1.coordinates[:, 1].max(), sig2.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_resolution),
            np.linspace(y_min, y_max, grid_resolution)
        )
        
        val1_grid = griddata(
            sig1.coordinates, sig1.raw_values,
            (grid_x, grid_y), method='linear', fill_value=np.nan
        )
        val2_grid = griddata(
            sig2.coordinates, sig2.raw_values,
            (grid_x, grid_y), method='linear', fill_value=np.nan
        )
        
        # Flatten and remove NaN
        mask = (~np.isnan(val1_grid)) & (~np.isnan(val2_grid))
        v1 = val1_grid[mask]
        v2 = val2_grid[mask]
        
        if len(v1) < 10:
            return 0.0, 0.0
        
        pearson_corr, _ = pearsonr(v1, v2)
        spearman_corr, _ = spearmanr(v1, v2)
        
        return pearson_corr if not np.isnan(pearson_corr) else 0.0, \
               spearman_corr if not np.isnan(spearman_corr) else 0.0
    
    def find_matching_mz_features(
        self,
        gene: str,
        gene_signature: SpatialSignature,
        mz_signatures: Dict[str, SpatialSignature],
        top_k: int = 20,
        weight_embedding: float = 0.6,
        weight_attention: float = 0.2,
        weight_raw: float = 0.2
    ) -> pd.DataFrame:
        """
        Find m/z features that best match a gene's spatial pattern.
        
        Uses a weighted combination of:
        - Deep embedding similarity (primary)
        - Attention/importance overlap
        - Raw correlation (for validation)
        """
        matches = []
        
        for mz_name, mz_sig in mz_signatures.items():
            similarities = self.compute_embedding_similarity(gene_signature, mz_sig)
            
            # Combined score
            combined = (
                weight_embedding * similarities['embedding_cosine'] +
                weight_attention * similarities['attention_overlap'] +
                weight_raw * max(similarities['raw_pearson'], 0)  # Only positive correlations
            )
            
            matches.append({
                'gene': gene,
                'rna_sample': gene_signature.sample_id,
                'mz_feature': mz_name,
                'msi_sample': mz_sig.sample_id,
                'embedding_cosine': similarities['embedding_cosine'],
                'embedding_euclidean': similarities['embedding_euclidean'],
                'attention_overlap': similarities['attention_overlap'],
                'raw_pearson': similarities['raw_pearson'],
                'raw_spearman': similarities['raw_spearman'],
                'combined_score': combined
            })
        
        matches_df = pd.DataFrame(matches)
        matches_df = matches_df.sort_values('combined_score', ascending=False)
        
        return matches_df.head(top_k)
    
    def visualize_saliency_map(
        self,
        signature: SpatialSignature,
        title: str = None,
        save_path: str = None
    ):
        """
        Visualize the spatial importance/saliency map.
        
        This shows which regions the model considers important for this feature.
        """
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # 1. Raw values
        im1 = axes[0].scatter(
            signature.coordinates[:, 0],
            signature.coordinates[:, 1],
            c=signature.raw_values,
            cmap='viridis',
            s=20,
            alpha=0.8
        )
        axes[0].set_title(f'Raw Values\n{signature.feature_name}', fontweight='bold')
        axes[0].set_xlabel('X (μm)')
        axes[0].set_ylabel('Y (μm)')
        plt.colorbar(im1, ax=axes[0])
        
        # 2. Node importance (saliency)
        im2 = axes[1].scatter(
            signature.coordinates[:, 0],
            signature.coordinates[:, 1],
            c=signature.node_importance,
            cmap='hot',
            s=20,
            alpha=0.8
        )
        axes[1].set_title('Attention-Based Importance\n(Saliency Map)', fontweight='bold')
        axes[1].set_xlabel('X (μm)')
        axes[1].set_ylabel('Y (μm)')
        plt.colorbar(im2, ax=axes[1], label='Importance')
        
        # 3. Weighted values (importance * raw)
        weighted = signature.raw_values * signature.node_importance
        weighted = (weighted - weighted.min()) / (weighted.max() - weighted.min() + 1e-8)
        
        im3 = axes[2].scatter(
            signature.coordinates[:, 0],
            signature.coordinates[:, 1],
            c=weighted,
            cmap='plasma',
            s=20,
            alpha=0.8
        )
        axes[2].set_title('Importance-Weighted Values', fontweight='bold')
        axes[2].set_xlabel('X (μm)')
        axes[2].set_ylabel('Y (μm)')
        plt.colorbar(im3, ax=axes[2])
        
        if title:
            plt.suptitle(title, fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
        else:
            plt.show()
    
    def visualize_gene_mz_match(
        self,
        gene_sig: SpatialSignature,
        mz_sig: SpatialSignature,
        similarities: Dict[str, float],
        save_path: str = None
    ):
        """
        Comprehensive visualization of gene-m/z spatial match.
        """
        fig = plt.figure(figsize=(20, 15))
        
        # Create grid
        gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)
        
        # Row 1: Raw spatial patterns
        ax1 = fig.add_subplot(gs[0, 0])
        im1 = ax1.scatter(
            gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
            c=gene_sig.raw_values, cmap='viridis', s=30
        )
        ax1.set_title(f'Gene: {gene_sig.feature_name}\n({gene_sig.sample_id})', fontweight='bold')
        plt.colorbar(im1, ax=ax1)
        
        ax2 = fig.add_subplot(gs[0, 1])
        im2 = ax2.scatter(
            mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
            c=mz_sig.raw_values, cmap='viridis', s=10
        )
        ax2.set_title(f'm/z: {mz_sig.feature_name}\n({mz_sig.sample_id})', fontweight='bold')
        plt.colorbar(im2, ax=ax2)
        
        # Row 1: Importance maps
        ax3 = fig.add_subplot(gs[0, 2])
        im3 = ax3.scatter(
            gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
            c=gene_sig.node_importance, cmap='hot', s=30
        )
        ax3.set_title('Gene Importance', fontweight='bold')
        plt.colorbar(im3, ax=ax3)
        
        ax4 = fig.add_subplot(gs[0, 3])
        im4 = ax4.scatter(
            mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
            c=mz_sig.node_importance, cmap='hot', s=10
        )
        ax4.set_title('m/z Importance', fontweight='bold')
        plt.colorbar(im4, ax=ax4)
        
        # Row 2: Embeddings visualization (t-SNE or PCA of node embeddings)
        from sklearn.decomposition import PCA
        
        ax5 = fig.add_subplot(gs[1, 0:2])
        pca = PCA(n_components=2)
        gene_emb_2d = pca.fit_transform(gene_sig.node_embeddings)
        sc5 = ax5.scatter(
            gene_emb_2d[:, 0], gene_emb_2d[:, 1],
            c=gene_sig.node_importance, cmap='viridis', s=20, alpha=0.7
        )
        ax5.set_title('Gene Node Embeddings (PCA)', fontweight='bold')
        ax5.set_xlabel('PC1')
        ax5.set_ylabel('PC2')
        plt.colorbar(sc5, ax=ax5, label='Importance')
        
        ax6 = fig.add_subplot(gs[1, 2:4])
        mz_emb_2d = pca.fit_transform(mz_sig.node_embeddings)
        sc6 = ax6.scatter(
            mz_emb_2d[:, 0], mz_emb_2d[:, 1],
            c=mz_sig.node_importance, cmap='viridis', s=10, alpha=0.7
        )
        ax6.set_title('m/z Node Embeddings (PCA)', fontweight='bold')
        ax6.set_xlabel('PC1')
        ax6.set_ylabel('PC2')
        plt.colorbar(sc6, ax=ax6, label='Importance')
        
        # Row 3: Similarity metrics and correlation plot
        ax7 = fig.add_subplot(gs[2, 0:2])
        
        # Interpolate and plot correlation
        grid_res = 50
        x_min = min(gene_sig.coordinates[:, 0].min(), mz_sig.coordinates[:, 0].min())
        x_max = max(gene_sig.coordinates[:, 0].max(), mz_sig.coordinates[:, 0].max())
        y_min = min(gene_sig.coordinates[:, 1].min(), mz_sig.coordinates[:, 1].min())
        y_max = max(gene_sig.coordinates[:, 1].max(), mz_sig.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        gene_grid = griddata(gene_sig.coordinates, gene_sig.raw_values, (grid_x, grid_y), method='linear')
        mz_grid = griddata(mz_sig.coordinates, mz_sig.raw_values, (grid_x, grid_y), method='linear')
        
        mask = (~np.isnan(gene_grid)) & (~np.isnan(mz_grid))
        if mask.sum() > 0:
            ax7.scatter(gene_grid[mask], mz_grid[mask], alpha=0.3, s=10)
            ax7.set_xlabel('Gene Expression')
            ax7.set_ylabel('m/z Intensity')
            ax7.set_title(f'Spatial Correlation\nPearson: {similarities["raw_pearson"]:.3f}', fontweight='bold')
        
        # Metrics summary
        ax8 = fig.add_subplot(gs[2, 2:4])
        ax8.axis('off')
        
        metrics_text = f"""
╔══════════════════════════════════════════════════╗
║         SIMILARITY METRICS SUMMARY               ║
╠══════════════════════════════════════════════════╣
║                                                  ║
║  🔷 Deep Embedding Similarity                    ║
║     • Cosine Similarity:    {similarities['embedding_cosine']:>8.4f}            ║
║     • Euclidean Similarity: {similarities['embedding_euclidean']:>8.4f}            ║
║                                                  ║
║  🎯 Attention/Region Overlap                     ║
║     • Importance Overlap:   {similarities['attention_overlap']:>8.4f}            ║
║                                                  ║
║  📊 Raw Correlation (Baseline)                   ║
║     • Pearson:              {similarities['raw_pearson']:>8.4f}            ║
║     • Spearman:             {similarities['raw_spearman']:>8.4f}            ║
║                                                  ║
╠══════════════════════════════════════════════════╣
║  Gene:  {gene_sig.feature_name:<20} ({gene_sig.sample_id})       ║
║  m/z:   {mz_sig.feature_name:<20} ({mz_sig.sample_id})       ║
╚══════════════════════════════════════════════════╝
"""
        ax8.text(0.05, 0.95, metrics_text, transform=ax8.transAxes,
                fontsize=10, verticalalignment='top', family='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.suptitle(
            f'Spatial Pattern Match: {gene_sig.feature_name} ↔ m/z {mz_sig.feature_name}',
            fontsize=16, fontweight='bold', y=1.02
        )
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()
        else:
            plt.show()
    
    def analyze_attention_patterns(
        self,
        signatures: List[SpatialSignature],
        output_prefix: str = 'attention_analysis'
    ):
        """
        Analyze attention patterns across multiple features to identify
        consistently important regions.
        """
        print("\nAnalyzing attention patterns across features...")
        
        if not signatures:
            print("  No signatures provided")
            return
        
        # Stack all importance maps (after interpolation to common grid)
        grid_res = 100
        
        # Find global coordinate bounds
        all_coords = np.vstack([s.coordinates for s in signatures])
        x_min, x_max = all_coords[:, 0].min(), all_coords[:, 0].max()
        y_min, y_max = all_coords[:, 1].min(), all_coords[:, 1].max()
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        importance_maps = []
        feature_names = []
        
        for sig in signatures:
            imp_grid = griddata(
                sig.coordinates, sig.node_importance,
                (grid_x, grid_y), method='linear', fill_value=0
            )
            importance_maps.append(imp_grid)
            feature_names.append(f"{sig.feature_name}_{sig.sample_id}")
        
        importance_stack = np.stack(importance_maps, axis=0)
        
        # Average importance across features
        mean_importance = importance_stack.mean(axis=0)
        std_importance = importance_stack.std(axis=0)
        
        # Coefficient of variation (identifies consistently important regions)
        cv_importance = std_importance / (mean_importance + 1e-8)
        consistent_importance = mean_importance * (1 - cv_importance)
        consistent_importance = np.clip(consistent_importance, 0, None)
        
        # Visualization
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        
        im1 = axes[0, 0].imshow(mean_importance, origin='lower', cmap='hot', aspect='auto')
        axes[0, 0].set_title('Mean Importance Across Features', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        im2 = axes[0, 1].imshow(std_importance, origin='lower', cmap='Blues', aspect='auto')
        axes[0, 1].set_title('Importance Variability (Std)', fontweight='bold')
        plt.colorbar(im2, ax=axes[0, 1])
        
        im3 = axes[1, 0].imshow(consistent_importance, origin='lower', cmap='plasma', aspect='auto')
        axes[1, 0].set_title('Consistently Important Regions', fontweight='bold')
        plt.colorbar(im3, ax=axes[1, 0])
        
        # Threshold to find hotspots
        threshold = np.percentile(consistent_importance, 90)
        hotspots = consistent_importance > threshold
        
        im4 = axes[1, 1].imshow(hotspots.astype(float), origin='lower', cmap='Reds', aspect='auto')
        axes[1, 1].set_title('Attention Hotspots (Top 10%)', fontweight='bold')
        plt.colorbar(im4, ax=axes[1, 1])
        
        plt.suptitle(f'Attention Pattern Analysis\n({len(signatures)} features)', 
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        save_path = os.path.join(
            self.output_dir, 'attention_analysis',
            f'{output_prefix}_attention_summary.png'
        )
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  Saved attention analysis to {save_path}")
        
        return {
            'mean_importance': mean_importance,
            'std_importance': std_importance,
            'consistent_importance': consistent_importance,
            'hotspots': hotspots
        }
    
    def run_full_analysis(
        self,
        smooth_sigma: float = 50.0,
        top_k: int = 20,
        epochs: int = 150
    ) -> pd.DataFrame:
        """
        Run the complete enhanced analysis pipeline.
        """
        print("\n" + "="*70)
        print("ENHANCED GENE-TO-M/Z SPATIAL PATTERN MATCHING")
        print("Using Deep Spatial Embeddings with Attention")
        print("="*70)
        
        # Check gene availability
        gene_availability = self.check_gene_availability()
        
        # =====================================================================
        # PHASE 1: Train models
        # =====================================================================
        print("\n" + "-"*70)
        print("PHASE 1: Training Spatial Attention Models")
        print("-"*70)
        
        # Prepare RNA data for training
        print("\nPreparing RNA training data...")
        rna_train_data = {}
        
        for sample_id in list(self.rna_data.keys())[:4]:  # Use subset for training
            adata = self.rna_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                features = adata.X.toarray()
            else:
                features = adata.X.copy()
            
            # Use first 50 features for training
            features = features[:, :min(50, features.shape[1])]
            
            data = self.prepare_graph_data(coords, features, smooth_sigma)
            rna_train_data[sample_id] = data
            print(f"  {sample_id}: {data.x.shape}")
        
        # Train RNA model
        if rna_train_data:
            self.rna_model = self.train_spatial_model(
                rna_train_data, model_type='rna', epochs=epochs
            )
        
        # Prepare MSI data for training
        print("\nPreparing MSI training data...")
        msi_train_data = {}
        
        for sample_id in list(self.msi_data.keys())[:4]:  # Use subset for training
            adata = self.msi_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                features = adata.X.toarray()
            else:
                features = adata.X.copy()
            
            # Use first 50 features for training
            features = features[:, :min(50, features.shape[1])]
            
            data = self.prepare_graph_data(coords, features, smooth_sigma)
            msi_train_data[sample_id] = data
            print(f"  {sample_id}: {data.x.shape}")
        
        # Train MSI model
        if msi_train_data:
            self.msi_model = self.train_spatial_model(
                msi_train_data, model_type='msi', epochs=epochs
            )
        
        # =====================================================================
        # PHASE 2: Extract spatial signatures
        # =====================================================================
        print("\n" + "-"*70)
        print("PHASE 2: Extracting Spatial Signatures")
        print("-"*70)
        
        all_results = []
        all_gene_signatures = []
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Processing gene: {gene}")
            print(f"{'='*50}")
            
            available_samples = [s for s, avail in gene_availability[gene].items() if avail]
            
            if not available_samples:
                print(f"  {gene} not found in any samples, skipping...")
                continue
            
            for rna_sample in available_samples[:2]:  # Limit samples for demo
                print(f"\n  Extracting signature from {rna_sample}...")
                
                gene_sig = self.extract_gene_spatial_signature(
                    rna_sample, gene, self.rna_model, smooth_sigma
                )
                
                if gene_sig is None:
                    continue
                
                all_gene_signatures.append(gene_sig)
                
                # Visualize gene saliency map
                saliency_path = os.path.join(
                    self.output_dir, 'saliency_maps',
                    f'{gene}_{rna_sample}_saliency.png'
                )
                self.visualize_saliency_map(
                    gene_sig,
                    title=f'Saliency Map: {gene} ({rna_sample})',
                    save_path=saliency_path
                )
                
                # Match against MSI samples
                for msi_sample in list(self.msi_data.keys())[:4]:  # Limit for demo
                    print(f"    Matching against MSI sample {msi_sample}...")
                    
                    # Extract m/z signatures if not already done
                    if msi_sample not in self.mz_signatures or len(self.mz_signatures) == 0:
                        mz_sigs = self.extract_mz_spatial_signatures(
                            msi_sample, self.msi_model, smooth_sigma
                        )
                    else:
                        mz_sigs = {
                            name: sig 
                            for name, sample_sigs in self.mz_signatures.items()
                            for sample_id, sig in sample_sigs.items()
                            if sample_id == msi_sample
                        }
                    
                    if not mz_sigs:
                        continue
                    
                    # Find matches
                    matches_df = self.find_matching_mz_features(
                        gene, gene_sig, mz_sigs, top_k=top_k
                    )
                    
                    if len(matches_df) > 0:
                        all_results.append(matches_df)
                        
                        # Visualize top match
                        top_match = matches_df.iloc[0]
                        mz_sig = mz_sigs[top_match['mz_feature']]
                        
                        similarities = {
                            'embedding_cosine': top_match['embedding_cosine'],
                            'embedding_euclidean': top_match['embedding_euclidean'],
                            'attention_overlap': top_match['attention_overlap'],
                            'raw_pearson': top_match['raw_pearson'],
                            'raw_spearman': top_match['raw_spearman']
                        }
                        
                        viz_path = os.path.join(
                            self.output_dir, 'gene_visualizations',
                            f'{gene}_{rna_sample}_mz{top_match["mz_feature"]}_{msi_sample}.png'
                        )
                        self.visualize_gene_mz_match(
                            gene_sig, mz_sig, similarities, save_path=viz_path
                        )
                        
                        print(f"      Top match: m/z {top_match['mz_feature']} "
                              f"(score: {top_match['combined_score']:.3f})")
        
        # =====================================================================
        # PHASE 3: Aggregate and save results
        # =====================================================================
        print("\n" + "-"*70)
        print("PHASE 3: Aggregating Results")
        print("-"*70)
        
        if all_results:
            combined_results = pd.concat(all_results, ignore_index=True)
            combined_results = combined_results.sort_values('combined_score', ascending=False)
            
            # Save results
            output_file = os.path.join(self.output_dir, 'gene_to_mz_matches_enhanced.csv')
            combined_results.to_csv(output_file, index=False)
            print(f"\nResults saved to: {output_file}")
            
            # Analyze attention patterns
            if all_gene_signatures:
                self.analyze_attention_patterns(
                    all_gene_signatures,
                    output_prefix='all_genes'
                )
            
            # Summary
            print("\n" + "="*70)
            print("SUMMARY")
            print("="*70)
            
            for gene in AAD_TARGET_GENES:
                gene_results = combined_results[combined_results['gene'] == gene]
                if len(gene_results) > 0:
                    best = gene_results.iloc[0]
                    print(f"\n{gene}:")
                    print(f"  Best m/z: {best['mz_feature']}")
                    print(f"  Combined score: {best['combined_score']:.3f}")
                    print(f"  Embedding cosine: {best['embedding_cosine']:.3f}")
                    print(f"  Attention overlap: {best['attention_overlap']:.3f}")
                    print(f"  Raw Pearson: {best['raw_pearson']:.3f}")
            
            return combined_results
        
        return None
    
    def save_embeddings(self, prefix: str = 'spatial_embeddings'):
        """Save learned embeddings for later use"""
        print("\nSaving embeddings...")
        
        # Save gene embeddings
        gene_emb_path = os.path.join(self.output_dir, 'embeddings', f'{prefix}_genes.npz')
        gene_data = {}
        for gene, sample_sigs in self.gene_signatures.items():
            for sample_id, sig in sample_sigs.items():
                key = f"{gene}_{sample_id}"
                gene_data[f"{key}_global"] = sig.global_embedding
                gene_data[f"{key}_importance"] = sig.node_importance
        
        if gene_data:
            np.savez_compressed(gene_emb_path, **gene_data)
            print(f"  Saved gene embeddings to {gene_emb_path}")
        
        # Save m/z embeddings
        mz_emb_path = os.path.join(self.output_dir, 'embeddings', f'{prefix}_mz.npz')
        mz_data = {}
        for mz, sample_sigs in self.mz_signatures.items():
            for sample_id, sig in sample_sigs.items():
                key = f"{mz}_{sample_id}"
                mz_data[f"{key}_global"] = sig.global_embedding
                mz_data[f"{key}_importance"] = sig.node_importance
        
        if mz_data:
            np.savez_compressed(mz_emb_path, **mz_data)
            print(f"  Saved m/z embeddings to {mz_emb_path}")



In [11]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    """Main analysis pipeline"""
    print("="*70)
    print("ENHANCED GENE-TO-M/Z SPATIAL PATTERN MATCHING PIPELINE")
    print("With Attention-Based Spatial Importance Mapping")
    print("="*70)
    
    # Initialize enhanced matcher
    matcher = EnhancedGeneToMzMatcher(
        output_dir='./gene_to_mz_results_enhanced',
        embedding_dim=64,
        hidden_dim=128,
        num_layers=3,
        num_heads=4
    )
    
    # Load data
    matcher.load_all_data()
    
    # Run analysis
    results = matcher.run_full_analysis(
        smooth_sigma=50.0,
        top_k=20,
        epochs=150
    )
    
    # Save embeddings
    matcher.save_embeddings()
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE!")
    print("="*70)
    print(f"\nResults directory: {matcher.output_dir}/")
    print("  - gene_to_mz_matches_enhanced.csv: All matches with embedding metrics")
    print("  - gene_visualizations/: Gene-m/z match visualizations")
    print("  - saliency_maps/: Attention-based importance maps")
    print("  - attention_analysis/: Cross-feature attention patterns")
    print("  - embeddings/: Saved spatial embeddings for reuse")
    
    return matcher, results


In [ ]:
if __name__ == "__main__":
    matcher, results = main()

ENHANCED GENE-TO-M/Z SPATIAL PATTERN MATCHING PIPELINE
With Attention-Based Spatial Importance Mapping
Loading RNA-seq data...
  YC_1: (2112, 800)
  YC_2: (2775, 800)
  YC_3: (2808, 800)
  YC_4: (2725, 800)
  YAD_1: (2915, 800)
  YAD_2: (2960, 800)
  YAD_3: (2880, 800)
  YAD_4: (2939, 800)
  AC_1: (3065, 800)
  AC_2: (3054, 800)
  AC_3: (2892, 800)
  AC_4: (3002, 800)
  AAD_1: (2700, 800)
  AAD_2: (2171, 800)
  AAD_3: (1584, 800)
  AAD_4: (2438, 800)

Loading MSI data...
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

ENHANCED GENE-TO-M/Z SPATIAL PATTERN MATCHING
Using Deep Spatial Embeddings with Attention

Checking gene availability across samples...
  Mapt: 16/16 samples
  Thy1: 16/16 samples
  Pmch: 16/16 